# 04_Text_Preprocessing

## Objective

The goal of this notebook is to clean and prepare the extracted clinical notes for machine learning and NLP modeling.

This notebook will:

- load the extracted first-24-hour clinical notes
- inspect raw clinical text examples
- remove MIMIC-III de-identification placeholders
- normalize text formatting
- create a cleaned text column
- analyze document lengths
- prepare the dataset for TF-IDF and ClinicalBERT modeling

The final output will be a cleaned text dataset saved for downstream machine learning and NLP tasks.

In [1]:
# Code Starter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

## Step 1: Load Dataset

In [9]:
notes = pd.read_csv(PROCESSED_PATH + "first24h_notes.csv")
print(notes.shape)
notes.head()

(36395, 5)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,SEPSIS_LABEL,TEXT
0,3,145834.0,211552,1,[**2101-10-20**] 10:23 PM\n CHEST (PORTABLE AP...
1,4,185777.0,294638,0,[**2191-3-16**] 0500\nGeneral: Pt in to EW fro...
2,6,107064.0,228232,0,2230-0700\n\nRecieved pt from PACU following L...
3,9,150750.0,220597,0,Respiratory Care:\n Pt. intubated in EW fo...
4,11,194540.0,229441,0,[**2178-4-16**] 2:49 PM\n CTA HEAD W&W/O C & R...


## Step 2: Inspect Raw Notes

In [86]:
pd.set_option("display.max_colwidth", 1000)

# To view what the notes actually look like
print(notes["TEXT"].iloc[0][:3000])

[**2101-10-20**] 10:23 PM
 CHEST (PORTABLE AP)                                             Clip # [**Clip Number (Radiology) 46359**]
 Reason: please eval for PTX, CHF, effusions, infiltrates
 ______________________________________________________________________________
 [**Hospital 2**] MEDICAL CONDITION:
     76 year old man with hypotension, vfib arrest, status post RSC line
  placement.
 REASON FOR THIS EXAMINATION:
  please eval for PTX, CHF, effusions, infiltrates
 ______________________________________________________________________________
                                 FINAL REPORT
 INDICATIONS:  Hypertension, V-Fib rest, s/p right subclavian line placement.

 PORTABLE CHEST:  Comparison is made to previous films from  four hours prior.

 FINDINGS: There has been placement of a right sided subclavian catheter, with
 the tip in the proximal SVC. Additionally, there is a left sided PICC line
 with the tip in the distal SVC. There has been interval intubation with
 endotrache

In [72]:
# To view some more notes
#for i in range(3):
    #print(f"\n{"=" * 100}")
    #print(notes["TEXT"].iloc[i][:3000])

## Step 3: Check Current Text Length

In [23]:
notes["TEXT_LENGTH"] = notes["TEXT"].str.len()

notes["TEXT_LENGTH"].describe()

count     36395.000000
mean      12295.214700
std       19021.550843
min           7.000000
25%        3044.000000
50%        5391.000000
75%       10523.500000
max      273734.000000
Name: TEXT_LENGTH, dtype: float64

## Step 4: Create Cleaning Function

In [27]:
import re 

def clean_clinical_text(text): 
    
    # Convert to string
    text = str(text)

    # Remove MIMIC de-identification placeholders
    text = re.sub(r"\[\*\*.*?\*\*\]", " ", text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading or trailing spaces
    text = text.strip()

    return text

Why Am I Removing These?

MIMIC notes contain things like:

In [40]:
"""

[**Clip_Number**]

[**Hospital**]

[**Date**]

"""

'\n\n[**Clip_Number**]\n\n[**Hospital**]\n\n[**Date**]\n\n'

These are artificial placeholders and provide no predictive value.

## Step 5: Create Clean Text Column

In [45]:
notes["CLEAN_TEXT"] = notes["TEXT"].apply(clean_clinical_text)

## Step 6: Compare Before vs After


In [67]:
print("RAW TEXT:\n")
print(notes["TEXT"].iloc[0][:3000])

print("\n")

print("CLEAN TEXT:\n")
print(notes["CLEAN_TEXT"].iloc[0][:3000])

RAW TEXT:

[**2101-10-20**] 10:23 PM
 CHEST (PORTABLE AP)                                             Clip # [**Clip Number (Radiology) 46359**]
 Reason: please eval for PTX, CHF, effusions, infiltrates
 ______________________________________________________________________________
 [**Hospital 2**] MEDICAL CONDITION:
     76 year old man with hypotension, vfib arrest, status post RSC line
  placement.
 REASON FOR THIS EXAMINATION:
  please eval for PTX, CHF, effusions, infiltrates
 ______________________________________________________________________________
                                 FINAL REPORT
 INDICATIONS:  Hypertension, V-Fib rest, s/p right subclavian line placement.

 PORTABLE CHEST:  Comparison is made to previous films from  four hours prior.

 FINDINGS: There has been placement of a right sided subclavian catheter, with
 the tip in the proximal SVC. Additionally, there is a left sided PICC line
 with the tip in the distal SVC. There has been interval intubation with


## Step 7: Check Clean Text Lengths

In [70]:
notes["CLEAN_TEXT_LENGTH"] = notes["CLEAN_TEXT"].str.len()

notes["CLEAN_TEXT_LENGTH"].describe()

count     36395.000000
mean      10196.443028
std       14973.986570
min           6.000000
25%        2769.500000
50%        4868.000000
75%        9327.000000
max      223335.000000
Name: CLEAN_TEXT_LENGTH, dtype: float64

## Step 8: Remove Empty Documents

In [77]:
# Sometimes cleaning leaves blank documents.

notes = notes[notes["CLEAN_TEXT"].str.len() > 0].copy()

print(notes.shape)

(36395, 8)


## Step 9: Save Clean Dataset

In [80]:
clean_notes = notes[[
    "SUBJECT_ID",
    "HADM_ID",
    "ICUSTAY_ID",
    "SEPSIS_LABEL",
    "CLEAN_TEXT"
    
]].copy()

clean_notes.to_csv(PROCESSED_PATH + "clean_notes.csv", index=False)

print("Saved Successfully!")

Saved Successfully!


## Summary

Clinical notes were cleaned and standardized for NLP analysis.

The preprocessing pipeline included:

- removing MIMIC-III de-identification placeholders
- converting text to lowercase
- normalizing whitespace
- removing empty documents

The final cleaned dataset contains 36,395 admission-level clinical documents and preserves the original sepsis class distribution.

This dataset will be used for feature extraction and machine learning model development in the next stage of the project.